# AI-Driven Market Analysis for Computer Component Price Surge

**Decision Support System for Hardware Procurement in the AI Era**

Pipeline: Scraping → Preprocessing → Statistical Analysis → Sentiment → AHP-TOPSIS → Visualization.

> Run cells top to bottom. CPU runtime is sufficient.

## 1. Environment Setup

In [1]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(
        ["git", "clone", "https://github.com/bugkey24/ai-era-pc-component-market-analysis.git"]
    )
    get_ipython().run_line_magic("cd", "ai-era-pc-component-market-analysis")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import nltk

nltk.download("stopwords", quiet=True)
print("Environment ready. IN_COLAB =", IN_COLAB)

Environment ready. IN_COLAB = False


## 2. Configuration & Imports

In [2]:
from src.utils import load_config, setup_logger

CONFIG_PATH = "config.yaml"
config = load_config(CONFIG_PATH)
logger = setup_logger(name="notebook", level=config["logging"]["level"])
config["scraping"]["platforms"]

[{'name': 'tokopedia',
  'enabled': True,
  'method': 'static',
  'base_url': 'https://www.tokopedia.com/search'},
 {'name': 'shopee',
  'enabled': True,
  'method': 'dynamic',
  'base_url': 'https://shopee.co.id/search'},
 {'name': 'blibli',
  'enabled': True,
  'method': 'static',
  'base_url': 'https://www.blibli.com/search'}]

## 3. Option A — Load Existing Data

Skip live scraping (platform markup changes often). Place CSVs in `data/raw/` or use the sample generator below.

In [3]:
from pathlib import Path

import pandas as pd

raw_dir = Path("data/raw")
csvs = sorted(raw_dir.glob("*.csv"))

if csvs:
    data = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    print(f"Loaded {len(data)} rows from {len(csvs)} file(s)")
else:
    # Minimal sample so the pipeline is runnable end-to-end offline
    import numpy as np

    rng = np.random.default_rng(42)
    rows = []
    for cat, base, n in [("gpu", 8_000_000, 20), ("ram", 1_400_000, 20), ("ssd", 800_000, 20)]:
        for i in range(n):
            rows.append(
                {
                    "product_id": cat.upper() + "-" + str(i).zfill(3),
                    "name": f"Sample {cat.upper()} Model {i} {rng.choice([chr(56)+chr(71)+chr(66), chr(49)+chr(54)+chr(71)+chr(66), chr(53)+chr(49)+chr(50)+chr(71)+chr(66), chr(49)+chr(84)+chr(66)])}",
                    "category": cat,
                    "price": f"Rp {base * rng.uniform(0.6, 1.6):,.0f}",
                    "rating": round(rng.uniform(3.8, 5.0), 1),
                    "review_count": int(rng.integers(5, 400)),
                    "seller_rating": round(rng.uniform(4.0, 5.0), 1),
                    "seller_followers": int(rng.integers(10, 5000)),
                    "source": str(rng.choice(["tokopedia", "shopee", "blibli"])),
                }
            )
    data = pd.DataFrame(rows)
    print(f"Generated {len(data)} sample rows (no raw CSVs found)")
data.head()

Generated 60 sample rows (no raw CSVs found)


,product_id,name,category,price,rating,review_count,seller_rating,seller_followers,source
0,GPU-000,Sample GPU Model 0 8GB,gpu,"Rp 8,311,028",4.8,310,4.7,1015,tokopedia
1,GPU-001,Sample GPU Model 1 512GB,gpu,"Rp 10,889,118",4.7,390,4.1,4200,shopee
2,GPU-002,Sample GPU Model 2 512GB,gpu,"Rp 12,214,120",4.6,151,4.8,2731,shopee
3,GPU-003,Sample GPU Model 3 16GB,gpu,"Rp 9,236,678",3.9,94,4.8,1391,shopee
4,GPU-004,Sample GPU Model 4 8GB,gpu,"Rp 7,636,208",5.0,304,4.9,3392,blibli


## 4. Option B — Live Scraping

⚠️ Platform markup changes frequently; selectors in `src/scrapers/` may need updating. Rate-limited per `config.yaml`.

In [4]:
# Uncomment to scrape live:
# pipeline = PipelineOrchestrator(CONFIG_PATH)
# data = pipeline._run_scraping()
# data.head()

## 5. Preprocessing & Feature Engineering

In [5]:
from src.preprocessing import DataPreprocessor, FeatureEngineer

clean = (
    DataPreprocessor(data)
    .clean_prices()
    .handle_missing(config["preprocessing"].get("handle_missing", "drop"))
    .extract_specifications()
    .remove_outliers(threshold=config["preprocessing"].get("outlier_threshold", 3.0))
)
fe = FeatureEngineer(clean.df)
clean = fe.create_price_per_gb().create_weighted_rating().create_seller_trust_score()
df = clean.get_engineered_data()
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

60 rows, 15 columns


,product_id,name,category,price,rating,review_count,seller_rating,seller_followers,source,spec_capacity,spec_memory_type,spec_interface,price_per_gb,weighted_rating,seller_trust
0,GPU-000,Sample GPU Model 0 8GB,gpu,8311028,4.8,310,4.7,1015,tokopedia,8GB,,,1038878.50,4.724,32.541
1,GPU-001,Sample GPU Model 1 512GB,gpu,10889118,4.7,390,4.1,4200,shopee,512GB,,,21267.81,4.641,34.207
2,GPU-002,Sample GPU Model 2 512GB,gpu,12214120,4.6,151,4.8,2731,shopee,512GB,,,23855.70,4.453,37.981
3,GPU-003,Sample GPU Model 3 16GB,gpu,9236678,3.9,94,4.8,1391,shopee,16GB,,,577292.38,3.703,34.745
4,GPU-004,Sample GPU Model 4 8GB,gpu,7636208,5.0,304,4.9,3392,blibli,8GB,,,954526.00,4.919,39.834


## 6. Statistical Analysis

In [6]:
from src.analysis import StatisticalAnalyzer

stats_an = StatisticalAnalyzer(df)
summary = stats_an.describe().get_summary()
display(summary.round(2))
stats_an.correlation_matrix()
display(stats_an.price_trend_by_category())
stats_an.normality_test("price")

,count,mean,std,min,25%,50%,75%,max,median,iqr,skewness,kurtosis
price,60.0,3797741.97,3820158.69,499888.00,1121845.25,1620108.00,7745832.50,12540078.00,1620108.00,6623987.25,0.96,-0.68
rating,60.0,4.38,0.34,3.90,4.10,4.40,4.62,5.00,4.40,0.53,0.15,-1.17
review_count,60.0,204.08,117.14,14.00,87.50,211.50,310.50,390.00,211.50,223.00,-0.10,-1.48
seller_rating,60.0,4.45,0.31,4.00,4.20,4.40,4.70,5.00,4.40,0.50,0.17,-1.36
seller_followers,60.0,2448.85,1383.40,120.00,1343.50,2557.00,3746.25,4694.00,2557.00,2402.75,0.07,-1.21
price_per_gb,60.0,128573.47,265565.91,488.17,1948.31,11719.48,121881.34,1322359.38,11719.48,119933.04,3.02,9.30
weighted_rating,60.0,4.18,0.39,3.24,3.91,4.17,4.46,4.92,4.17,0.55,-0.30,-0.26
seller_trust,60.0,33.59,4.15,20.62,31.02,34.20,36.42,40.60,34.20,5.40,-0.66,0.63


,mean,median,std,min,max,count,cv
category,,,,,,,
gpu,8882314.35,8502547.0,1986553.31,4858898,12540078,20,0.224
ram,1664349.65,1620108.0,359056.09,1123309,2181983,20,0.216
ssd,846561.90,834598.0,263387.14,499888,1263657,20,0.311


{'statistic': 0.7652, 'p_value': 0.0, 'normal_at_0.05': False}

## 7. Sentiment Analysis

The SVM trains automatically when review CSVs exist in data/raw/reviews_*.csv (produced by the review scrapers; the pipeline merges per-product scores). Labels are weak supervision from review ratings. If the corpus is skewed (e-commerce reality: mostly positive), the model fits on full data and accuracy is reported as not measurable. A demo set below keeps this notebook runnable standalone.

In [7]:
from src.analysis import SentimentAnalyzer

demo_texts = [
    "harga mahal sekali",
    "terlalu mahal untuk spesifikasi ini",
    "harga naik terus",
    "harga murah dan terjangkau",
    "murah bagus worth it",
    "harga oke murah",
    "performa cepat stabil",
    "cepat dan stabil untuk AI training",
    "performa mantap",
    "lambat dan sering hang",
    "performa jelek lambat",
    "lemot sering ngehang",
] * 3
demo_labels = (["negative"] * 3 + ["positive"] * 3 + ["positive"] * 3 + ["negative"] * 3) * 3

sentiment = SentimentAnalyzer(language=config["sentiment"]["language"])
sentiment.train(demo_texts, demo_labels)
print(f"Demo-model accuracy: {sentiment.accuracy:.2f}  (replace with real labelled reviews)")

Demo-model accuracy: 1.00  (replace with real labelled reviews)


## 8. AHP-TOPSIS Decision Model

In [8]:
import numpy as np

from src.dss import AHPProcessor, TOPSISProcessor

dss_cfg = config["dss"]

ahp = AHPProcessor(dss_cfg["criteria"])
ahp.build_pairwise_matrix(dss_cfg["pairwise_matrix"])
ahp.calculate_weights().check_consistency()
print(ahp.summary())
assert ahp.is_consistent(), "Pairwise matrix inconsistent (CR >= 0.1) — revise config"

{'criteria': ['price', 'performance', 'rating', 'seller_reliability', 'sentiment', 'future_value'], 'weights': {'price': np.float64(0.227), 'performance': np.float64(0.4387), 'rating': np.float64(0.0918), 'seller_reliability': np.float64(0.0413), 'sentiment': np.float64(0.0413), 'future_value': np.float64(0.1598)}, 'lambda_max': 6.3633, 'consistency_ratio': 0.0586, 'is_consistent': True}


In [9]:
# Decision matrix: map config criteria to available columns
col_map = {
    "price": "price",
    "performance": "rating",
    "rating": "weighted_rating",
    "seller_reliability": "seller_trust",
    "sentiment": "rating",  # placeholder until review-level sentiment exists
    "future_value": "price_per_gb",
}
matrix = np.column_stack(
    [pd.to_numeric(df[col_map.get(c, c)], errors="coerce").fillna(0) for c in dss_cfg["criteria"]]
)

topsis = TOPSISProcessor(matrix, ahp.get_weights(), dss_cfg["criteria_types"])
ranking = topsis.rank()
df_ranked = (
    df.reset_index(drop=True)
    .loc[ranking["Alternative"]]
    .assign(Score=ranking["Score"].values, Rank=ranking["Rank"].values)
)
df_ranked[["name", "category", "price", "rating", "Score", "Rank"]].head(10)

,name,category,price,rating,Score,Rank
1,Sample GPU Model 1 512GB,gpu,10889118,4.7,0.1188,1
55,Sample SSD Model 15 1TB,ssd,734393,4.4,0.4102,2
58,Sample SSD Model 18 8GB,ssd,521569,4.9,0.4318,3
43,Sample SSD Model 3 8GB,ssd,584846,3.9,0.4237,4
2,Sample GPU Model 2 512GB,gpu,12214120,4.6,0.0849,5
53,Sample SSD Model 13 512GB,ssd,707776,4.0,0.4077,6
59,Sample SSD Model 19 1TB,ssd,1263657,4.8,0.4031,7
48,Sample SSD Model 8 16GB,ssd,1117454,4.1,0.4133,8
57,Sample SSD Model 17 1TB,ssd,519403,4.2,0.4129,9
46,Sample SSD Model 6 512GB,ssd,815291,5.0,0.4142,10


## 9. Visualization

In [10]:
from src.visualization import Visualizer

viz = Visualizer(
    df,
    output_dir="outputs/visualizations",
    **{k: v for k, v in config["visualization"].items() if k in ("style", "palette", "dpi")},
)
_ = viz.plot_price_trends()
_ = viz.plot_correlation_heatmap()
_ = viz.plot_ranking_bar_chart(ranking)
print("Charts saved to outputs/visualizations/")

Charts saved to outputs/visualizations/


## 10. Export Results

In [11]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)
df.to_csv(out / "cleaned_data.csv", index=False)
ranking.to_csv(out / "rankings.csv", index=False)
print("Saved:", [p.name for p in out.iterdir()])

Saved: ['cleaned_data.csv', 'rankings.csv', 'visualizations']


## 11. Conclusions

- **Why prices rose:** fill in after running against live data.
- **Top value pick:** see ranking table above.
- **Normalization outlook:** see `docs/03-methodology.md` Phase 6 scenarios.

*Replace the demo sample/sentiment data with real scraped data and labelled reviews for production-grade results.*